In [ ]:
!pip install -q efficientnet

In [ ]:
%%capture
!pip install wandb

In [ ]:
# Asthetics

import warnings
import sklearn.exceptions
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=sklearn.exceptions.UndefinedMetricWarning)

# General
from kaggle_datasets import KaggleDatasets
from glob import glob
import pandas as pd
import numpy as np
import os
import time
import cv2
import random
import shutil
import math
import re
pd.set_option('display.max_columns', None)

# Visualizations
from PIL import Image
from plotly.subplots import make_subplots
from plotly.offline import iplot
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import wandb
import plotly.graph_objs as go
import plotly.figure_factory as ff
import plotly.express as px
%matplotlib inline
sns.set(style="whitegrid")

# Machine Learning
# Pre Procesing
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
# Models
from sklearn.model_selection import train_test_split, KFold
# Deep Learning
import tensorflow as tf
import tensorflow.keras.backend as K
import efficientnet.tfkeras as efn
from wandb.keras import WandbCallback
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
import tensorflow_addons as tfa
from tensorflow_addons.metrics import F1Score, FBetaScore
from tensorflow_addons.callbacks import TQDMProgressBar
from tensorflow.keras.utils import plot_model

#Metrics
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print('TF',tf.__version__)

# Random Seed Fixing
RANDOM_SEED = 42

def seed_everything(seed=RANDOM_SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)

seed_everything()

In [ ]:
# From https://www.kaggle.com/xhlulu/ranzcr-efficientnet-tpu-training
def auto_select_accelerator():
    TPU_DETECTED = False
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.experimental.TPUStrategy(tpu)
        print("Running on TPU:", tpu.master())
        TPU_DETECTED =True
    except ValueError:
        strategy = tf.distribute.get_strategy()
    print(f"Running on {strategy.num_replicas_in_sync} replicas")
    
    return strategy, TPU_DETECTED

# CFG

In [ ]:
strategy, TPU_DETECTED = auto_select_accelerator()
AUTO = tf.data.experimental.AUTOTUNE
REPLICAS = strategy.num_replicas_in_sync

In [ ]:
CFG = {
    'version': 0,
    'feature_name': 'cqt',
    'fold': 0,
    'EFFV': 7,
    'size': 256,
    'dropout': 0.3,
    'label_smoothing': 0.,
    'batch_size': 64,
    'steps_multiplier': 0.25,
    'epochs': 30,
    'aug': True,
    'MIX_UP_P': 0.1,
    'S_SHIFT': 0.,
    'T_SHIFT': 0.,
    'R_ANGLE': 0. / 180 * np.pi,
    'opt': 'AdamW',
    'lr': 1e-3,
    'lr_start': 1e-4,
    'lr_max': 0.000015 * REPLICAS * 64,
    'lr_min': 1e-7,
    'lr_ramp_ep': 4,
    'lr_sus_ep': 0,
    'lr_decay': 0.70
}

In [ ]:
from tqdm.notebook import tqdm

files_train_g = []
for i,k in tqdm([(0, 1), (2, 3), (4,5), (6, 7), (8, 9) ,(10,11), (12, 13), (14, 15)]):
    GCS_PATH = KaggleDatasets().get_gcs_path(f'cqt-g2net-v2-{i}-{k}')
    files_train_g.extend(np.sort(np.array(tf.io.gfile.glob(GCS_PATH + '/train*.tfrec'))).tolist())
num_train_files = len(files_train_g)
print('train_files:',num_train_files)

# Reading Tfrecords

In [ ]:

def mixup(image, label, PROBABILITY = 1.0, AUG_BATCH=CFG['batch_size'] * REPLICAS):
    # input image - is a batch of images of size [n,dim,dim,3] not a single image of [dim,dim,3]
    # output - a batch of images with mixup applied
    DIM = CFG['size']
    
    imgs = []; labs = []
    for j in range(AUG_BATCH):
        # DO MIXUP WITH PROBABILITY DEFINED ABOVE
        P = tf.cast( tf.random.uniform([],0,1)<=PROBABILITY, tf.float32)
        # CHOOSE RANDOM
        k = tf.cast( tf.random.uniform([],0,AUG_BATCH),tf.int32)
        a = tf.random.uniform([],0,1)*P # this is beta dist with alpha=1.0
        # MAKE MIXUP IMAGE
        img1 = image[j,]
        img2 = image[k,]
        imgs.append((1-a)*img1 + a*img2)
        # MAKE CUTMIX LABEL
        lab1 = label[j,]
        lab2 = label[k,]
        labs.append((1-a)*lab1 + a*lab2)
            
    # RESHAPE HACK SO TPU COMPILER KNOWS SHAPE OF OUTPUT TENSOR (maybe use Python typing instead?)
    image2 = tf.reshape(tf.stack(imgs),(AUG_BATCH,DIM,DIM,3))
    label2 = tf.reshape(tf.stack(labs),(AUG_BATCH, 1))
    return image2,label2

def time_shift(img, shift=CFG['T_SHIFT']):
    if shift > 0:
        T = CFG['size']
        P = tf.random.uniform([],0,1)
        SHIFT = tf.cast(T * P, tf.int32)
        return tf.concat([img[-SHIFT:], img[:-SHIFT]], axis=0)
    return img


def spector_shift(img, shift=CFG['S_SHIFT']):
    if shift > 0:
        T = CFG['size']
        P = tf.random.uniform([],0,1)
        SHIFT = tf.cast(T * P, tf.int32)
        return tf.concat([img[:, -SHIFT:], img[:, :-SHIFT]], axis=1)
    return img

def rotate(img, angle=CFG['R_ANGLE']):
    if angle > 0:
        P = tf.random.uniform([],0,1)
        A = tf.cast(angle * P, tf.float32)
        return tfa.image.rotate(img, A)
    return img
    
def img_aug_f(img):
    img = time_shift(img)
    img = spector_shift(img)
    img = rotate(img)
    return img

def imgs_aug_f(imgs, batch_size):
    _imgs = []
    DIM = CFG['size']
    for j in range(batch_size):
        _imgs.append(img_aug_f(imgs[j]))
    return tf.reshape(tf.stack(_imgs),(batch_size,DIM,DIM,3))

def aug_f(imgs, labels, batch_size):
    imgs, label = mixup(imgs, labels, CFG['MIX_UP_P'], batch_size)
    imgs = imgs_aug_f(imgs, batch_size)
    return imgs, labels

def read_labeled_tfrecord(example):
    tfrec_format = {
        'image'                        : tf.io.FixedLenFeature([], tf.string),
        'image_id'                     : tf.io.FixedLenFeature([], tf.string),
        'target'                       : tf.io.FixedLenFeature([], tf.int64)
    }           
    example = tf.io.parse_single_example(example, tfrec_format)
    label = tf.cast(example['target'], tf.float32)
    return prepare_image(example['image']), label


def read_unlabeled_tfrecord(example, return_image_id):
    tfrec_format = {
        'image'                        : tf.io.FixedLenFeature([], tf.string),
        'image_id'                     : tf.io.FixedLenFeature([], tf.string),
    }
    example = tf.io.parse_single_example(example, tfrec_format)
    return prepare_image(example['image']), example['image_id'] if return_image_id else 0

 
def prepare_image(img, dim=CFG['size']):    
    img = tf.image.resize(tf.image.decode_png(img, channels=3), size=(dim, dim))
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.reshape(img, [dim,dim, 3])
            
    return img

def count_data_items(fileids):
    n = [int(re.compile(r"-([0-9]*)\.").search(fileid).group(1)) 
         for fileid in fileids]
    return np.sum(n)

# Dataset Creation

In [ ]:
def get_dataset(files, shuffle = False, repeat = False, 
                labeled=True, return_image_ids=True, batch_size=16, dim=CFG['size'], aug=False):
    
    ds = tf.data.TFRecordDataset(files, num_parallel_reads=AUTO)
    ds = ds.cache()
    
    if repeat:
        ds = ds.repeat()
    
    if shuffle: 
        ds = ds.shuffle(1024*2)
        opt = tf.data.Options()
        opt.experimental_deterministic = False
        ds = ds.with_options(opt)
        
    if labeled: 
        ds = ds.map(read_labeled_tfrecord, num_parallel_calls=AUTO)
    else:
        ds = ds.map(lambda example: read_unlabeled_tfrecord(example, return_image_ids), 
                    num_parallel_calls=AUTO)      
    
    ds = ds.batch(batch_size * REPLICAS)
    if aug:
        ds = ds.map(lambda x, y: aug_f(x, y, batch_size * REPLICAS), num_parallel_calls=AUTO)
    ds = ds.prefetch(AUTO)
    return ds

# Build Model

In [ ]:
import tensorflow as tf
import math


class AngularGrad(tf.keras.optimizers.Optimizer):
    def __init__(
          self,
          method_angle: str = "cos",
          learning_rate=1e-3,
          beta_1=0.9,
          beta_2=0.999,
          eps=1e-7,
          name: str = "AngularGrad",
          **kwargs,
      ):
        super().__init__(name, **kwargs)

        self.method_angle = method_angle
        self._set_hyper("learning_rate", kwargs.get("lr", learning_rate))
        self._set_hyper("beta_1", beta_1)
        self._set_hyper("beta_2", beta_2)
        self._set_hyper("eps", eps)
        self.eps = eps or tf.keras.backend.epsilon()

    def _create_slots(self, var_list):
        for var in var_list:
            self.add_slot(var, "exp_avg")
            self.add_slot(var, "exp_avg_sq")
            self.add_slot(var, "previous_grad")
            self.add_slot(var, "min", initializer=tf.keras.initializers.Constant(value=math.pi / 2))
            self.add_slot(var, "final_angle_function_theta")

    def _resource_apply_dense(self, grad, var):
        var_dtype = var.dtype.base_dtype

        lr = self._get_hyper("learning_rate", var_dtype)
        beta_1 = self._get_hyper("beta_1", var_dtype)
        beta_2 = self._get_hyper("beta_2", var_dtype)
        eps = self._get_hyper("eps", var_dtype)

        exp_avg = self.get_slot(var, "exp_avg")
        exp_avg_sq = self.get_slot(var, "exp_avg_sq")
        previous_grad = self.get_slot(var, "previous_grad")
        min = self.get_slot(var, "min")
        final_angle_function_theta = self.get_slot(var, "final_angle_function_theta")

        step = tf.cast(self.iterations + 1, var_dtype)
        beta_1_power = tf.pow(beta_1, step)
        beta_2_power = tf.pow(beta_2, step)

        new_exp_avg = exp_avg.assign(
            beta_1 * exp_avg + (1.0 - beta_1) * grad,
            use_locking=self._use_locking
        )
        exp_avg_corrected = new_exp_avg / (1.0 - beta_1_power)

        new_exp_avg_sq = exp_avg_sq.assign(
            beta_2 * exp_avg_sq + (1.0 - beta_2) * tf.square(grad),
            use_locking=self._use_locking,
        )
        exp_avg_sq_corrected = new_exp_avg_sq / (1.0 - beta_2_power)

        tan_theta = tf.abs((previous_grad - grad) / (1 + previous_grad * grad))
        cos_theta = 1 / tf.sqrt(1 + tf.square(tan_theta))

        angle = tf.atan(tan_theta) * (180 / math.pi)
        ans = tf.greater(angle, min)
        mean_ans = tf.reduce_mean(tf.cast(ans, tf.float32))

        def true_fn():
            new_min = min.assign(angle, use_locking=self._use_locking)
            new_final_angle_function_theta = final_angle_function_theta.assign(
            tf.identity(tan_theta if self.method_angle == "tan" else cos_theta),
              use_locking=self._use_locking
              )
            return new_min, new_final_angle_function_theta

        def false_fn():
            return min, final_angle_function_theta

        new_min, new_final_angle_function_theta = tf.cond(tf.less(mean_ans, 0.5), true_fn, false_fn)
        angular_coeff = tf.tanh(tf.abs(final_angle_function_theta)) * 0.5 + 0.5

        var_update = var.assign_sub(
            lr * exp_avg_corrected * angular_coeff / (tf.sqrt(exp_avg_sq_corrected) + eps),
            use_locking=self._use_locking
        )

        new_previous_grad = previous_grad.assign(grad, use_locking=self._use_locking)

        updates = [var_update, new_exp_avg, new_exp_avg_sq, new_min, new_previous_grad, new_final_angle_function_theta]
        return tf.group(*updates)

    def _resource_apply_sparse(self, grad, var, indices):
        raise NotImplementedError

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "learning_rate": self._serialize_hyperparameter("learning_rate"),
                "beta_1": self._serialize_hyperparameter("beta_1"),
                "beta_2": self._serialize_hyperparameter("beta_2"),
                "eps": self._serialize_hyperparameter("eps")
            }
        )
        return config

In [ ]:
def get_init_lr_fun(config=CFG):
    lr_start   = config['lr_start'] * 0.5
    lr_max     = config['lr_max'] * 0.5
    lr_min     = config['lr_min'] * 0.5
    lr_ramp_ep = config['lr_ramp_ep']
    lr_sus_ep  = config['lr_sus_ep']
    lr_decay   = config['lr_decay']
   
    def lrfn(epoch):
        if epoch < lr_ramp_ep:
            lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
            
        elif epoch < lr_ramp_ep + lr_sus_ep:
            lr = lr_max
            
        else:
            lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
            
        return lr
    return lrfn

def get_max_lr_fun(config=CFG):
    lr_start   = config['lr_start']
    lr_max     = config['lr_max']
    lr_min     = config['lr_min']
    lr_ramp_ep = config['lr_ramp_ep']
    lr_sus_ep  = config['lr_sus_ep']
    lr_decay   = config['lr_decay']
   
    def lrfn(epoch):
        if epoch < lr_ramp_ep:
            lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
            
        elif epoch < lr_ramp_ep + lr_sus_ep:
            lr = lr_max
            
        else:
            lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
            
        return lr
    return lrfn

In [ ]:
EFNS = [efn.EfficientNetB0, efn.EfficientNetB1, efn.EfficientNetB2, efn.EfficientNetB3, 
        efn.EfficientNetB4, efn.EfficientNetB5, efn.EfficientNetB6, efn.EfficientNetB7]

def build_model(config, count=820):
    inp = tf.keras.layers.Input(shape=(config['size'], config['size'],3))
    base = EFNS[config['EFFV']](input_shape=(config['size'],config['size'],3),weights='imagenet',include_top=False)
    
    x = base(inp)
    
    x = tf.keras.layers.GlobalAvgPool2D()(x)
    x = tf.keras.layers.Dropout(config['dropout'])(x)
    
    x = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs=inp, outputs=x)

    if config['opt'] == 'AngularGrad':
        opt = AngularGrad(config['lr'])
    else:
        lr_decayed_fn = tf.keras.optimizers.schedules.PolynomialDecay(
                            config['lr'], count, end_learning_rate=1e-5, power=2.0,
                            cycle=True
                            )

        opt = tfa.optimizers.AdamW(lr_decayed_fn, learning_rate=config['lr'])
    loss = tf.keras.losses.BinaryCrossentropy(label_smoothing=config['label_smoothing']) 
    model.compile(optimizer=opt,loss=loss,metrics=['AUC'])
    return model

# Training

In [ ]:
def get_lr_callback(config=CFG):
    lr_start   = config['lr_start']
    lr_max     = config['lr_max']
    lr_min     = config['lr_min']
    lr_ramp_ep = config['lr_ramp_ep']
    lr_sus_ep  = config['lr_sus_ep']
    lr_decay   = config['lr_decay']
   
    def lrfn(epoch):
        if epoch < lr_ramp_ep:
            lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
            
        elif epoch < lr_ramp_ep + lr_sus_ep:
            lr = lr_max
            
        else:
            lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
            
        return lr

    lr_callback = tf.keras.callbacks.LearningRateScheduler(lrfn, verbose=False)
    return lr_callback

In [ ]:
def vis_lr_callback(config=CFG):
    lr_start   = config['lr_start']
    lr_max     = config['lr_max']
    lr_min     = config['lr_min']
    lr_ramp_ep = config['lr_ramp_ep']
    lr_sus_ep  = config['lr_sus_ep']
    lr_decay   = config['lr_decay']
   
    def lrfn(epoch):
        if epoch < lr_ramp_ep:
            lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
            
        elif epoch < lr_ramp_ep + lr_sus_ep:
            lr = lr_max
        else:
            lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
            
        return lr
    plt.figure(figsize=(10, 7))
    plt.plot([lrfn(i) for i in range(config['epochs'])])
    plt.show()

In [ ]:
vis_lr_callback()

In [ ]:
skf = KFold(n_splits=4,shuffle=True,random_state=2809)
oof_pred = []; oof_tar = []; oof_val = []; oof_f1 = []; oof_ids = []; oof_folds = [] 

files_train_g = np.array(files_train_g)

for fold,(idxT,idxV) in enumerate(skf.split(files_train_g)):
    # CREATE TRAIN AND VALIDATION SUBSETS
    files_train = files_train_g[idxT]
    np.random.shuffle(files_train);
    files_valid = files_train_g[idxV]
    CFG['fold'] = fold
    
    run = wandb.init(project='g2net',
                     config=CFG,
                     mode='offline'
                    )
    config = wandb.config
    print('#'*25); print('#### FOLD',fold+1)
    print('#### Image Size: %i | model: %s | batch_size %i'%
          (config['size'], EFNS[config['EFFV']].__name__,config['batch_size']*REPLICAS))
    train_images = count_data_items(files_train)
    val_images   = count_data_items(files_valid)
    print('#### Training: %i | Validation: %i'%(train_images, val_images))
    
    # BUILD MODEL
    K.clear_session()
    with strategy.scope():
        model = build_model(config, count=int(count_data_items(files_train)/config['batch_size']//REPLICAS*config['steps_multiplier']))
    print('#'*25)   
    # SAVE BEST MODEL EACH FOLD
    sv = tf.keras.callbacks.ModelCheckpoint(
        'fold-%i.h5'%fold, monitor='val_auc', verbose=0, save_best_only=True,
        save_weights_only=True, mode='max', save_freq='epoch')
   
    # TRAIN
    print('Training...')
    history = model.fit(
        get_dataset(files_train, shuffle=True, repeat=True,
                dim=config['size'], batch_size = config['batch_size'], aug=config['aug']), 
        epochs=CFG['epochs'], 
        callbacks = [sv, get_lr_callback(), WandbCallback()], 
        steps_per_epoch=int(count_data_items(files_train)/config['batch_size']//REPLICAS*config['steps_multiplier']),
        validation_data=get_dataset(files_valid, shuffle=False,
                repeat=False,dim=config['size']),
        verbose=1
    )
    
    # Loading best model for inference
    print('Loading best model...')
    model.load_weights('fold-%i.h5'%fold)  
    
    ds_valid = get_dataset(files_valid,labeled=True,return_image_ids=False,
            repeat=False,shuffle=False,dim=CFG['size'],batch_size=CFG['batch_size']*2)
    model.evaluate(ds_valid,verbose=1)
    wandb.save('fold-%i.h5'%fold)
    run.join()

# Datasets
* [Q-Transform TFRecords](https://www.kaggle.com/miklgr500/q-transform-tfrecords)
    * [CQT G2Net V2 [0 - 1]](https://www.kaggle.com/miklgr500/cqt-g2net-v2-0-1)
    * [CQT G2Net V2 [2 - 3]](https://www.kaggle.com/miklgr500/cqt-g2net-v2-2-3)
    * [CQT G2Net V2 [4 - 5]](https://www.kaggle.com/miklgr500/cqt-g2net-v2-4-5)
    * [CQT G2Net V2 [6 - 7]](https://www.kaggle.com/miklgr500/cqt-g2net-v2-6-7)
    * [CQT G2Net V2 [8 - 9]](https://www.kaggle.com/miklgr500/cqt-g2net-v2-8-9)
    * [CQT G2Net V2 [10 - 11]](https://www.kaggle.com/miklgr500/cqt-g2net-v2-10-11)
    * [CQT G2Net V2 [12 - 13]](https://www.kaggle.com/miklgr500/cqt-g2net-v2-12-13)
    * [CQT G2Net V2 [14 - 15]](https://www.kaggle.com/miklgr500/cqt-g2net-v2-14-15)

# Next steps
* Generate Test Sets
* Create Inference Notebook
* Add augmentation
* Add TTA Inference